In [1]:
import pandas as pd
import numpy as np

In [2]:
# Ingest the fully featured master dataset
df = pd.read_parquet('../01_data/04_engineered/shipment_features.parquet')

In [3]:
# ROUTE AGGREGATION PIPELINE

# Aggregate operational metrics to evaluate origin-destination path performance
route_analytics = df.groupby('route_state').agg(
    total_shipments=('order_id', 'count'),
    avg_lead_time=('shipping_lead_time', 'mean'),
    std_lead_time=('shipping_lead_time', 'std'),
    delayed_shipments=('is_delayed', 'sum'),
    total_sales=('sales', 'sum'),
    total_profit=('gross_profit', 'sum')
).reset_index()

In [4]:
# Derive Route Efficiency Score (RES) representing strict on-time delivery proportion
route_analytics['route_efficiency_score'] = (
    (route_analytics['total_shipments'] - route_analytics['delayed_shipments'])
    / route_analytics['total_shipments'] * 100
).round(2)

In [5]:
# Calculate Coefficient of Variation (CV) to quantify shipment transit predictability
route_analytics['lead_time_cv'] = (
    route_analytics['std_lead_time'] / route_analytics['avg_lead_time']
).fillna(0).round(4)

In [6]:
# CARRIER TIER & REGIONAL DRILL-DOWN

# Assess carrier class degradation patterns across geographical zones
ship_mode_analytics = df.groupby(['ship_mode', 'region']).agg(
    volume=('order_id', 'count'),
    avg_lead_time=('shipping_lead_time', 'mean'),
    actual_delay_rate=('is_delayed', 'mean')
).reset_index()

ship_mode_analytics['actual_delay_rate'] = (ship_mode_analytics['actual_delay_rate'] * 100).round(2)

In [7]:
# PIPELINE VALIDATION & EXPORT
print("\n--- RUNTIME QUALITY ASSURANCE ---")
assert len(route_analytics) == df['route_state'].nunique(), "Validation Failed: Route mismatch!"
print(f"Validation Passed: 100% of the {df['route_state'].nunique()} unique network routes aggregated successfully.")
print(f"Global Network Baseline Metric: {route_analytics['route_efficiency_score'].mean():.2f}% Route Efficiency.")


--- RUNTIME QUALITY ASSURANCE ---
Validation Passed: 100% of the 196 unique network routes aggregated successfully.
Global Network Baseline Metric: 53.50% Route Efficiency.


In [8]:
# Persist analytical summaries for downstream UI/Visualization ingestion
route_analytics.to_parquet('../01_data/04_engineered/summary_route_intelligence.parquet', index=False)
ship_mode_analytics.to_parquet('../01_data/04_engineered/summary_ship_mode.parquet', index=False)

print("\nSUCCESS: Stage 04 Analytics Completed. Summaries registered in '01_data/04_engineered/'")


SUCCESS: Stage 04 Analytics Completed. Summaries registered in '01_data/04_engineered/'
